# Planners-9-HTN - Planification Hierarchique

**Navigation** : [Index](../../README.md) | [<< Temporal](Planners-8-Temporal.ipynb) | [LLM-Planning >>](../04-NeuroSymbolic/Planners-10-LLM-Planning.ipynb)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :

1. **Comprendre** le paradigme de planification hierarchique (HTN)
2. **Distinguer** les taches abstraites des taches primitives
3. **Definir** des methodes de decomposition avec preconditions
4. **Implementer** un solveur HTN simplifie
5. **Comparer** HTN avec la planification classique STRIPS

### Prerequis

- Notebooks Planners-1 a 3 maitrises (PDDL, espaces d'etats)
- Notebook Planners-4 (Fast Downward) recommande
- Python 3.9+ avec unified-planning

### Duree estimee : 45 minutes

---

## 1. Introduction a la Planification Hierarchique

La **planification hierarchique** (Hierarchical Task Network - HTN) est un paradigme de planification qui structure le probleme en taches pouvant etre decomposees recursivement. Contrairement a la planification classique qui cherche un chemin dans un espace d'etats, HTN cherche a **decomposer une tache complexe** en actions primitives.

### 1.1 Motivation

Dans la planification classique STRIPS/PDDL :
- L'objectif est un **etat but** (conjonction de predicats)
- Le planificateur explore l'espace d'etats
- Le domaine est "plat" : toutes les actions sont au meme niveau

En planification HTN :
- L'objectif est une **tache a accomplir**
- Le planificateur decompose les taches
- Le domaine est **hierarchique** : taches abstraites et primitives

### 1.2 Analogie : Planifier un voyage

**Planification classique** : On specifie l'etat final desire
```
But: a(Tokyo) ^ avoir_billet(Paris, Tokyo)
```

**Planification HTN** : On specifie la tache a accomplir
```
Tache: voyager(Paris, Tokyo)
  -> Methode 1: voyager_en_avion
       sous-taches: [reserver_billet, aller_aeroport, prendre_vol, sortir_aeroport]
  -> Methode 2: voyager_en_train (si distance < 1000km)
       sous-taches: [acheter_billet, aller_gare, prendre_train]
```

La **connaissance de l'expert** sur "comment voyager" est encodee dans les methodes.

### 1.3 Comparaison conceptuelle

| Aspect | Planification classique | Planification HTN |
|--------|------------------------|-------------------|
| **Objectif** | Etat but a atteindre | Tache a accomplir |
| **Connaissance** | Actions atomiques | Methodes de decomposition |
| **Recherche** | Dans l'espace d'etats | Dans l'espace des decompositions |
| **Expertise** | Encodee implicitement | Encodee explicitement |
| **Flexibilite** | Haute (n'importe quel plan) | Guidee (plans "raisonnables") |

---

## 2. Fondements Theoriques des HTN

### 2.1 Composantes d'un domaine HTN

Un domaine HTN comprend :

1. **Predicats** : Faits sur l'etat du monde (comme en PDDL)
2. **Actions primitives** : Actions executables avec preconditions et effets
3. **Taches abstraites** : Taches non-executables directement
4. **Methodes** : Regles de decomposition des taches abstraites

```
Domaine HTN = <Predicats, Actions, Taches, Methodes>
```

### 2.2 Taches primitives vs abstraites

**Tache primitive** : Correspond a une action executable
- A des preconditions et des effets
- Est directement executable dans le monde
- Exemple : `!drive(truck, loc-a, loc-b)`

**Tache abstraite** : Requiert une decomposition
- N'est pas directement executable
- Peut etre decomposee de multiples facons (methodes)
- Exemple : `deliver(package, loc-a, loc-b)`

> **Notation** : Par convention, les taches primitives sont souvent prefixees par `!`.

### 2.3 Methodes de decomposition

Une **methode** definit comment decomposer une tache abstraite :

$$m = \langle \text{name}, \text{task}, \text{precond}, \text{subtasks} \rangle$$

- **name** : Identifiant de la methode
- **task** : Tache abstraite a decomposer
- **precond** : Preconditions pour que cette methode soit applicable
- **subtasks** : Liste ordonnee de sous-taches (reseau de taches)

#### Exemple : Methode pour livrer un colis

```
Methode: m_deliver_by_truck
  Tache: deliver(pkg, from, to)
  Preconditions: 
    - truck-available(t)
    - connected(from, to)
  Sous-taches:
    1. !load(pkg, t, from)
    2. !drive(t, from, to)
    3. !unload(pkg, t, to)
```

### 2.4 Reseaux de taches (Task Networks)

Un **reseau de taches** est un ensemble partiellement ordonne de taches avec des contraintes :

- **Taches** : Ensemble de taches (primitives ou abstraites)
- **Ordre** : Contraintes de precedence ($t_1 \prec t_2$)
- **Contraintes** : Relations sur les variables

```
Task Network pour "construire-maison":
  Taches: {fondation, murs, toit, fenetres}
  Ordre: fondation < murs < toit, murs < fenetres
```

La decomposition progressive du reseau de taches initial produit un plan.

---

## 3. HDDL - Hierarchical Domain Definition Language

**HDDL** est une extension de PDDL pour la planification hierarchique. Il ajoute la syntaxe pour definir des taches et des methodes.

### 3.1 Structure d'un domaine HDDL

```lisp
(define (domain logistics-htn)
  (:requirements :strips :typing :hierarchy)  ;; :hierarchy pour HTN
  
  ;; Types et predicats (comme PDDL)
  (:types location package vehicle)
  (:predicates (at ?p - package ?l - location) ...)
  
  ;; Actions primitives
  (:action drive :parameters (?v - vehicle ?from ?to - location) ...)
  
  ;; Taches (declarees explicitement)
  (:task deliver :parameters (?p - package ?from ?to - location))
  
  ;; Methodes
  (:method m_deliver
    :parameters (?p - package ?from ?to - location ?t - vehicle)
    :task (deliver ?p ?from ?to)
    :precondition (and (at-vehicle ?t ?from) ...)
    :subtasks (and 
               (load ?p ?t ?from)
               (drive ?t ?from ?to)
               (unload ?p ?t ?to))
  )
)
```

### 3.2 Exemple complet : Domaine Logistics HTN

In [1]:
# Definition du domaine HDDL pour Logistics
HDDL_DOMAIN = """
(define (domain logistics-htn)
  (:requirements :strips :typing :hierarchy)
  (:types location package vehicle)
  
  (:predicates
    (at-pkg ?p - package ?l - location)
    (at-veh ?v - vehicle ?l - location)
    (in ?p - package ?v - vehicle)
    (connected ?from ?to - location)
  )
  
  ;; ========== Actions primitives ==========
  
  (:action load
    :parameters (?p - package ?v - vehicle ?l - location)
    :precondition (and (at-pkg ?p ?l) (at-veh ?v ?l))
    :effect (and (in ?p ?v) (not (at-pkg ?p ?l)))
  )
  
  (:action unload
    :parameters (?p - package ?v - vehicle ?l - location)
    :precondition (in ?p ?v)
    :effect (and (at-pkg ?p ?l) (not (in ?p ?v)))
  )
  
  (:action drive
    :parameters (?v - vehicle ?from ?to - location)
    :precondition (and (at-veh ?v ?from) (connected ?from ?to))
    :effect (and (at-veh ?v ?to) (not (at-veh ?v ?from)))
  )
  
  ;; ========== Taches abstraites ==========
  
  (:task deliver
    :parameters (?p - package ?from ?to - location)
  )
  
  ;; ========== Methodes ==========
  
  (:method m_deliver_direct
    :parameters (?p - package ?from ?to - location ?v - vehicle)
    :task (deliver ?p ?from ?to)
    :precondition (and 
                    (at-veh ?v ?from) 
                    (connected ?from ?to))
    :subtasks (and
                (task (load ?p ?v ?from))
                (task (drive ?v ?from ?to))
                (task (unload ?p ?v ?to))
              )
  )
  
  (:method m_deliver_via_hub
    :parameters (?p - package ?from ?hub ?to - location ?v1 ?v2 - vehicle)
    :task (deliver ?p ?from ?to)
    :precondition (and
                    (at-veh ?v1 ?from)
                    (at-veh ?v2 ?hub)
                    (connected ?from ?hub)
                    (connected ?hub ?to)
                    (not (= ?from ?hub))
                    (not (= ?hub ?to))
                  )
    :subtasks (and
                (task (load ?p ?v1 ?from))
                (task (drive ?v1 ?from ?hub))
                (task (unload ?p ?v1 ?hub))
                (task (load ?p ?v2 ?hub))
                (task (drive ?v2 ?hub ?to))
                (task (unload ?p ?v2 ?to))
              )
  )
)
"""

print("Domaine HDDL defini: logistics-htn")
print("\nActions primitives: load, unload, drive")
print("Taches abstraites: deliver")
print("Methodes: m_deliver_direct, m_deliver_via_hub")

Domaine HDDL defini: logistics-htn

Actions primitives: load, unload, drive
Taches abstraites: deliver
Methodes: m_deliver_direct, m_deliver_via_hub


### 3.3 Definition du probleme HDDL

Un probleme HDDL specifie le reseau de taches initial plutot qu'un etat but.

In [2]:
# Definition du probleme HDDL
HDDL_PROBLEM = """
(define (problem logistics-htn-1)
  (:domain logistics-htn)
  (:objects
    depot warehouse store - location
    pkg1 pkg2 - package
    truck1 - vehicle
  )
  
  (:init
    ;; Positions initiales des colis
    (at-pkg pkg1 depot)
    (at-pkg pkg2 warehouse)
    
    ;; Position initiale du camion
    (at-veh truck1 depot)
    
    ;; Connexions entre lieux
    (connected depot warehouse)
    (connected warehouse store)
    (connected depot store)
    (connected warehouse depot)
    (connected store warehouse)
    (connected store depot)
  )
  
  ;; Tache initiale (au lieu de :goal)
  (:htn
    :subtasks (and
                (task (deliver pkg1 depot store))
                (task (deliver pkg2 warehouse depot))
              )
  )
)
"""

print("Probleme HDDL defini: logistics-htn-1")
print("\nTaches initiales:")
print("  1. deliver(pkg1, depot, store)")
print("  2. deliver(pkg2, warehouse, depot)")

Probleme HDDL defini: logistics-htn-1

Taches initiales:
  1. deliver(pkg1, depot, store)
  2. deliver(pkg2, warehouse, depot)


### Interpretation du domaine HDDL

| Element | Description |
|---------|-------------|
| **Actions** | `load`, `unload`, `drive` - operations atomiques |
| **Tache abstraite** | `deliver(pkg, from, to)` - a decomposer |
| **m_deliver_direct** | Methode avec 1 vehicule, 3 sous-taches |
| **m_deliver_via_hub** | Methode avec 2 vehicules via hub, 6 sous-taches |

> **Note** : La methode `via_hub` est utile quand il n'y a pas de connexion directe ou quand differentes compagnies gerent differents segments.

---

## 4. Implementation d'un Solveur HTN Simplifie

Implementons un solveur HTN simple en Python pour comprendre les mecanismes de decomposition.

In [3]:
# Implementation d'un solveur HTN simplifie
from dataclasses import dataclass, field
from typing import List, Dict, Set, Tuple, Optional, Any, Union
from copy import deepcopy

@dataclass
class State:
    """Representation d'un etat du monde."""
    facts: Set[Tuple[str, ...]]  # Ensemble de faits (predicats)
    
    def holds(self, predicate: Tuple[str, ...]) -> bool:
        """Verifie si un fait est vrai dans l'etat."""
        return predicate in self.facts
    
    def add(self, predicate: Tuple[str, ...]) -> 'State':
        """Ajoute un fait et retourne un nouvel etat."""
        new_facts = self.facts | {predicate}
        return State(new_facts)
    
    def remove(self, predicate: Tuple[str, ...]) -> 'State':
        """Retire un fait et retourne un nouvel etat."""
        new_facts = self.facts - {predicate}
        return State(new_facts)
    
    def __repr__(self):
        return f"State({sorted(self.facts)})"


@dataclass
class Action:
    """Action primitive avec preconditions et effets."""
    name: str
    parameters: List[str]
    preconditions: List[Tuple[str, ...]]
    add_effects: List[Tuple[str, ...]]
    del_effects: List[Tuple[str, ...]]
    
    def is_applicable(self, state: State, binding: Dict[str, str]) -> bool:
        """Verifie si l'action est applicable avec les bindings donnes."""
        for precond in self.preconditions:
            ground = tuple(binding.get(p, p) for p in precond)
            if not state.holds(ground):
                return False
        return True
    
    def apply(self, state: State, binding: Dict[str, str]) -> State:
        """Applique l'action et retourne le nouvel etat."""
        new_state = state
        for eff in self.del_effects:
            ground = tuple(binding.get(p, p) for p in eff)
            new_state = new_state.remove(ground)
        for eff in self.add_effects:
            ground = tuple(binding.get(p, p) for p in eff)
            new_state = new_state.add(ground)
        return new_state
    
    def ground(self, binding: Dict[str, str]) -> str:
        """Retourne la representation groundee de l'action."""
        params = [binding.get(p, p) for p in self.parameters]
        return f"{self.name}({', '.join(params)})"


@dataclass
class Method:
    """Methode de decomposition pour tache abstraite."""
    name: str
    task_name: str
    task_params: List[str]
    parameters: List[str]
    preconditions: List[Tuple[str, ...]]
    subtasks: List[Tuple[str, List[str]]]  # (nom_tache, params)
    
    def is_applicable(self, state: State, binding: Dict[str, str]) -> bool:
        """Verifie si la methode est applicable."""
        for precond in self.preconditions:
            ground = tuple(binding.get(p, p) for p in precond)
            if not state.holds(ground):
                return False
        return True

print("Classes de base definies: State, Action, Method")

Classes de base definies: State, Action, Method


### 4.1 Classes de base pour HTN

Les classes `State`, `Action`, et `Method` forment le coeur de notre representation HTN :

| Classe | Role |
|--------|------|
| **State** | Represente l'etat du monde comme un ensemble de faits |
| **Action** | Action primitive avec preconditions et effets (add/delete) |
| **Method** | Regle de decomposition avec preconditions et sous-taches |

Le code suivant definit ces structures de donnees.

In [4]:
# Solveur HTN simplifie
@dataclass
class HTNPlanner:
    """Solveur HTN avec decomposition ordonnee."""
    actions: Dict[str, Action]
    methods: Dict[str, List[Method]]
    
    def solve(self, initial_state: State, 
              tasks: List[Tuple[str, List[str]]],
              binding: Dict[str, str] = None) -> Optional[List[str]]:
        """
        Resout le probleme HTN par decomposition.
        
        Args:
            initial_state: Etat initial
            tasks: Liste de taches (primitives ou abstraites)
            binding: Bindings des variables
        
        Returns:
            Liste d'actions groundees ou None si echec
        """
        if binding is None:
            binding = {}
        
        return self._decompose(initial_state, tasks, binding, [])
    
    def _decompose(self, state: State, 
                   tasks: List[Tuple[str, List[str]]],
                   binding: Dict[str, str],
                   plan: List[str]) -> Optional[List[str]]:
        """Decomposition recursive des taches."""
        
        # Cas de base: plus de taches
        if not tasks:
            return plan
        
        current_task, current_params = tasks[0]
        remaining_tasks = tasks[1:]
        
        # Ground les parametres
        grounded_params = [binding.get(p, p) for p in current_params]
        
        # Cas 1: Tache primitive
        if current_task in self.actions:
            action = self.actions[current_task]
            action_binding = {p: v for p, v in zip(action.parameters, grounded_params)}
            
            if action.is_applicable(state, action_binding):
                new_state = action.apply(state, action_binding)
                action_str = action.ground(action_binding)
                return self._decompose(new_state, remaining_tasks, binding, plan + [action_str])
            else:
                # Action non applicable - echec
                return None
        
        # Cas 2: Tache abstraite - essayer les methodes
        if current_task in self.methods:
            for method in self.methods[current_task]:
                # Creer le binding pour les parametres de la tache
                method_binding = binding.copy()
                for tp, gp in zip(method.task_params, grounded_params):
                    method_binding[tp] = gp
                
                # Trouver les bindings pour les parametres supplementaires de la methode
                # (ceux qui ne sont pas dans les parametres de la tache)
                task_param_set = set(method.task_params)
                extra_params = [p for p in method.parameters if p not in task_param_set]
                
                if extra_params:
                    # Chercher tous les objets dans l'etat qui pourraient convenir
                    all_objects = set()
                    for fact in state.facts:
                        all_objects.update(fact[1:])  # Tous les objets (sauf le predicat)
                    
                    # Essayer chaque combinaison d'objets pour les parametres supplementaires
                    for obj in all_objects:
                        extended_binding = method_binding.copy()
                        extended_binding[extra_params[0]] = obj
                        
                        # Verifier les preconditions avec ce binding etendu
                        if method.is_applicable(state, extended_binding):
                            # Ground les sous-taches
                            grounded_subtasks = []
                            for st_name, st_params in method.subtasks:
                                grounded = [extended_binding.get(p, p) for p in st_params]
                                grounded_subtasks.append((st_name, grounded))
                            
                            # Essayer avec cette decomposition
                            new_tasks = grounded_subtasks + remaining_tasks
                            result = self._decompose(state, new_tasks, extended_binding, plan)
                            if result is not None:
                                return result
                else:
                    # Pas de parametres supplementaires
                    if method.is_applicable(state, method_binding):
                        # Ground les sous-taches
                        grounded_subtasks = []
                        for st_name, st_params in method.subtasks:
                            grounded = [method_binding.get(p, p) for p in st_params]
                            grounded_subtasks.append((st_name, grounded))
                        
                        # Essayer avec cette decomposition
                        new_tasks = grounded_subtasks + remaining_tasks
                        result = self._decompose(state, new_tasks, method_binding, plan)
                        if result is not None:
                            return result
            
            # Aucune methode n'a fonctionne
            return None
        
        # Tache inconnue
        print(f"ERREUR: Tache inconnue: {current_task}")
        return None

print("Solveur HTN defini: HTNPlanner")

Solveur HTN defini: HTNPlanner


---

## 5. Exemple Pratique : Logistics avec HTN

Appliquons notre solveur HTN au domaine Logistics.

In [5]:
# Definition du domaine HTN pour Logistics

# Actions primitives
actions = {
    'load': Action(
        name='load',
        parameters=['?p', '?v', '?l'],
        preconditions=[('at-pkg', '?p', '?l'), ('at-veh', '?v', '?l')],
        add_effects=[('in', '?p', '?v')],
        del_effects=[('at-pkg', '?p', '?l')]
    ),
    'unload': Action(
        name='unload',
        parameters=['?p', '?v', '?l'],
        preconditions=[('in', '?p', '?v'), ('at-veh', '?v', '?l')],
        add_effects=[('at-pkg', '?p', '?l')],
        del_effects=[('in', '?p', '?v')]
    ),
    'drive': Action(
        name='drive',
        parameters=['?v', '?from', '?to'],
        preconditions=[('at-veh', '?v', '?from'), ('connected', '?from', '?to')],
        add_effects=[('at-veh', '?v', '?to')],
        del_effects=[('at-veh', '?v', '?from')]
    )
}

# Methodes pour la tache abstraite 'deliver'
methods = {
    'deliver': [
        Method(
            name='m_deliver_direct',
            task_name='deliver',
            task_params=['?p', '?from', '?to'],
            parameters=['?p', '?from', '?to', '?v'],
            preconditions=[
                ('at-veh', '?v', '?from'),
                ('connected', '?from', '?to')
            ],
            subtasks=[
                ('load', ['?p', '?v', '?from']),
                ('drive', ['?v', '?from', '?to']),
                ('unload', ['?p', '?v', '?to'])
            ]
        ),
        Method(
            name='m_deliver_via_intermediate',
            task_name='deliver',
            task_params=['?p', '?from', '?to'],
            parameters=['?p', '?from', '?mid', '?to', '?v'],
            preconditions=[
                ('at-veh', '?v', '?from'),
                ('connected', '?from', '?mid'),
                ('connected', '?mid', '?to')
            ],
            subtasks=[
                ('load', ['?p', '?v', '?from']),
                ('drive', ['?v', '?from', '?mid']),
                ('drive', ['?v', '?mid', '?to']),
                ('unload', ['?p', '?v', '?to'])
            ]
        )
    ]
}

print("Domaine HTN Logistics defini")
print(f"Actions: {list(actions.keys())}")
print(f"Methodes pour 'deliver': {[m.name for m in methods['deliver']]}")

Domaine HTN Logistics defini
Actions: ['load', 'unload', 'drive']
Methodes pour 'deliver': ['m_deliver_direct', 'm_deliver_via_intermediate']


Definition du probleme logistique concret : etat initial avec les positions des colis et vehicules, but a atteindre, puis resolution par decomposition HTN.

In [6]:
# Definition du probleme et resolution

# Etat initial
initial_facts = {
    # Positions des colis
    ('at-pkg', 'pkg1', 'depot'),
    ('at-pkg', 'pkg2', 'warehouse'),
    
    # Position du camion
    ('at-veh', 'truck1', 'depot'),
    
    # Connexions (graphe non oriente)
    ('connected', 'depot', 'warehouse'),
    ('connected', 'warehouse', 'depot'),
    ('connected', 'warehouse', 'store'),
    ('connected', 'store', 'warehouse'),
    ('connected', 'depot', 'store'),
    ('connected', 'store', 'depot'),
}

initial_state = State(initial_facts)

# Tache unique a accomplir (pour simplifier l'exemple)
tasks = [
    ('deliver', ['pkg1', 'depot', 'store']),
]

# Creation du solveur
planner = HTNPlanner(actions=actions, methods=methods)

# Resolution
print("=" * 60)
print("RESOLUTION HTN - Exemple 1")
print("=" * 60)
print(f"\nEtat initial:")
print(f"  pkg1 au depot, pkg2 au warehouse")
print(f"  truck1 au depot")
print(f"\nTache:")
print(f"  1. deliver(pkg1, depot, store)")
print("\n" + "-" * 60)

plan = planner.solve(initial_state, tasks)

if plan:
    print("\nPLAN TROUVE:")
    print("=" * 60)
    for i, action in enumerate(plan, 1):
        print(f"  {i}. {action}")
    print("=" * 60)
    print(f"Longueur du plan: {len(plan)} actions")
else:
    print("ECHEC: Aucun plan trouve")

# Deuxieme exemple: livrer depuis le warehouse
print("\n\n" + "=" * 60)
print("RESOLUTION HTN - Exemple 2")
print("=" * 60)

# Pour livrer pkg2 depuis warehouse, on doit d'abord deplacer le camion
# Ce n'est pas possible avec nos methodes actuelles
print(f"\nRemarque: Le deuxieme colis pkg2 est au warehouse.")
print(f"Apres la premiere livraison, truck1 est au 'store'.")
print(f"Pour livrer pkg2, il faudrait une methode de repositionnement.")

RESOLUTION HTN - Exemple 1

Etat initial:
  pkg1 au depot, pkg2 au warehouse
  truck1 au depot

Tache:
  1. deliver(pkg1, depot, store)

------------------------------------------------------------

PLAN TROUVE:
  1. load(pkg1, truck1, depot)
  2. drive(truck1, depot, store)
  3. unload(pkg1, truck1, store)
Longueur du plan: 3 actions


RESOLUTION HTN - Exemple 2

Remarque: Le deuxieme colis pkg2 est au warehouse.
Apres la premiere livraison, truck1 est au 'store'.
Pour livrer pkg2, il faudrait une methode de repositionnement.


### 5.1 Definition du domaine HTN Logistics

Nous definissons maintenant les actions primitives (`load`, `unload`, `drive`) et les methodes de decomposition pour la tache abstraite `deliver`.

### Interpretation du plan HTN

Le solveur HTN a decompose la tache abstraite en actions primitives :

| Etape | Action | Etat resultat |
|-------|--------|---------------|
| 1 | `load(pkg1, truck1, depot)` | pkg1 est dans truck1 |
| 2 | `drive(truck1, depot, store)` | truck1 est maintenant au store |
| 3 | `unload(pkg1, truck1, store)` | pkg1 est au store |

**Analyse de la decomposition** :
1. La methode `m_deliver_direct` a ete choisie (preconditions satisfaites)
2. Les sous-taches ont ete executees sequentiellement
3. Chaque action primitive verifie ses preconditions avant execution

**Limitation de ce solveur simplifie** :
- Le solveur ne gere pas le repositionnement automatique du vehicule
- Pour livrer pkg2 (depuis warehouse), il faudrait que le camion y soit deja
- Un solveur HTN complet aurait une methode `reposition_vehicle` pour gerer ce cas

---

## 6. Algorithme SHOP2

**SHOP2** (Simple Hierarchical Ordered Planner 2) est l'un des planificateurs HTN les plus connus. Il utilise une strategie de decomposition ordonnee.

### 6.1 Principe de SHOP2

SHOP2 fonctionne selon le principe **Ordered Task Decomposition** :

1. Maintenir une liste ordonnee de taches
2. Traiter la premiere tache de la liste
3. Si primitive : verifier preconditions, appliquer, passer a la suivante
4. Si abstraite : choisir une methode applicable, remplacer par ses sous-taches

```
SHOP2(state, tasks):
    si tasks est vide:
        retourner []  # succes
    
    t = tasks[0]  # premiere tache
    rest = tasks[1:]  # reste des taches
    
    si t est primitive:
        si t applicable dans state:
            new_state = appliquer(t, state)
            return SHOP2(new_state, rest)
        sinon:
            retourner ECHEC
    
    si t est abstraite:
        pour chaque methode m applicable pour t:
            subtasks = decomposer(m, t)
            resultat = SHOP2(state, subtasks + rest)
            si resultat != ECHEC:
                retourner resultat
        retourner ECHEC
```

### 6.2 Proprietes de SHOP2

| Propriete | Description |
|-----------|-------------|
| **Completude** | Si un plan existe, SHOP2 le trouvera (avec backtrack) |
| **Efficacite** | Guide par les methodes, exploration reduite |
| **Ordered** | Traite les taches en ordre (premiere d'abord) |
| **State-based** | Verifie l'etat courant pour choisir les methodes |

In [7]:
# Visualisation de la decomposition HTN
def visualize_htn_decomposition():
    """
    Affiche l'arbre de decomposition HTN pour notre exemple.
    """
    tree = """
    Arbre de decomposition HTN pour deliver(pkg1, depot, store)
    ============================================================
    
                     deliver(pkg1, depot, store)
                              |
                    [m_deliver_direct selectionnee]
                              |
           +------------------+------------------+
           |                  |                  |
       load(pkg1,        drive(truck1,      unload(pkg1,
       truck1, depot)    depot, store)      truck1, store)
           |                  |                  |
       [PRIMITIVE]        [PRIMITIVE]        [PRIMITIVE]
    
    
    Selection de la methode:
    ------------------------
    - m_deliver_direct: Preconditions OK (truck1 au depot, depot-store connectes)
    - m_deliver_via_intermediate: Non essayee (direct suffisante)
    
    """
    print(tree)

visualize_htn_decomposition()


    Arbre de decomposition HTN pour deliver(pkg1, depot, store)

                     deliver(pkg1, depot, store)
                              |
                    [m_deliver_direct selectionnee]
                              |
           +------------------+------------------+
           |                  |                  |
       load(pkg1,        drive(truck1,      unload(pkg1,
       truck1, depot)    depot, store)      truck1, store)
           |                  |                  |
       [PRIMITIVE]        [PRIMITIVE]        [PRIMITIVE]


    Selection de la methode:
    ------------------------
    - m_deliver_direct: Preconditions OK (truck1 au depot, depot-store connectes)
    - m_deliver_via_intermediate: Non essayee (direct suffisante)

    


---

## 7. Comparaison HTN vs Planification Classique

### 7.1 Tableau comparatif

| Critere | Planification Classique | Planification HTN |
|---------|------------------------|-------------------|
| **Representation du but** | Etats a atteindre | Taches a accomplir |
| **Connaissance du domaine** | Actions primitives | Methodes de decomposition |
| **Espace de recherche** | Etats du monde | Decompositions possibles |
| **Taille de l'espace** | $O(b^d)$ | $O(m^h)$ ou m = methodes, h = hauteur |
| **Expertise encodee** | Implicite | Explicite (methodes) |
| **Plans generes** | Quelconque (optimal ou non) | "Raisonnable" selon l'expert |
| **Flexibilite** | Elevee | Contrainte par les methodes |

In [8]:
# Demonstration de la difference conceptuelle
def compare_planning_paradigms():
    """
    Compare les deux approches sur un exemple.
    """
    print("=" * 70)
    print("COMPARAISON: Planification Classique vs HTN")
    print("=" * 70)
    
    print("\n--- Planification Classique (PDDL) ---")
    print("""
    Probleme:
      Etat initial: at(pkg1, depot), at(truck1, depot)
      But: at(pkg1, store)
    
    Le planificateur explore l'espace d'etats:
      - Etat 0: {at(pkg1, depot), at(truck1, depot)}
      - Etat 1: {in(pkg1, truck1), at(truck1, depot)}  [load]
      - Etat 2: {in(pkg1, truck1), at(truck1, store)}  [drive]
      - Etat 3: {at(pkg1, store), at(truck1, store)}   [unload] <- BUT
    
    Avantage: Trouve le plan optimal (si heuristique admissible)
    Inconvenient: Peut explorer beaucoup d'etats inutiles
    """)
    
    print("\n--- Planification HTN ---")
    print("""
    Probleme:
      Etat initial: at(pkg1, depot), at(truck1, depot)
      Tache: deliver(pkg1, depot, store)
    
    Le planificateur decompose la tache:
      deliver(pkg1, depot, store)
        -> m_deliver_direct applicable (truck1 au depot)
        -> sous-taches: [load, drive, unload]
    
    Avantage: Exploite la connaissance de l'expert
    Inconvenient: Ne trouve que les plans prevus par les methodes
    """)

compare_planning_paradigms()

COMPARAISON: Planification Classique vs HTN

--- Planification Classique (PDDL) ---

    Probleme:
      Etat initial: at(pkg1, depot), at(truck1, depot)
      But: at(pkg1, store)

    Le planificateur explore l'espace d'etats:
      - Etat 0: {at(pkg1, depot), at(truck1, depot)}
      - Etat 1: {in(pkg1, truck1), at(truck1, depot)}  [load]
      - Etat 2: {in(pkg1, truck1), at(truck1, store)}  [drive]
      - Etat 3: {at(pkg1, store), at(truck1, store)}   [unload] <- BUT

    Avantage: Trouve le plan optimal (si heuristique admissible)
    Inconvenient: Peut explorer beaucoup d'etats inutiles
    

--- Planification HTN ---

    Probleme:
      Etat initial: at(pkg1, depot), at(truck1, depot)
      Tache: deliver(pkg1, depot, store)

    Le planificateur decompose la tache:
      deliver(pkg1, depot, store)
        -> m_deliver_direct applicable (truck1 au depot)
        -> sous-taches: [load, drive, unload]

    Avantage: Exploite la connaissance de l'expert
    Inconvenient: Ne tro

### 7.2 Quand utiliser HTN ?

**HTN est preferable quand** :
- Les "bonnes facons" de faire les choses sont connues
- L'expertise humaine peut etre encodee en methodes
- On veut des plans "raisonnables" plutot qu'optimaux
- Le domaine a une structure hierarchique naturelle

**Planification classique est preferable quand** :
- L'optimalite est critique
- Le domaine est difficile a encoder en methodes
- On veut explorer toutes les possibilites
- Aucune expertise n'est disponible

---

## 8. Exercices

### Exercice 1 : Ajouter une methode

Ajoutez une methode `m_deliver_with_transfer` pour le cas ou deux vehicules sont necessaires (transfert a un hub).

**Consignes** :
1. La methode doit avoir 4 sous-taches minimum
2. Les preconditions doivent verifier la disponibilite des deux vehicules
3. Testez avec un probleme approprie

In [9]:
# Exercice 1: Votre code ici

# Ajoutez la methode m_deliver_with_transfer
new_method = Method(
    name='m_deliver_with_transfer',
    task_name='deliver',
    task_params=['?p', '?from', '?to'],
    parameters=['?p', '?from', '?hub', '?to', '?v1', '?v2'],
    preconditions=[
        # TODO: Definir les preconditions
    ],
    subtasks=[
        # TODO: Definir les sous-taches
    ]
)

# Ajouter au dictionnaire des methodes et tester

### Exercice 2 : Comparaison de complexite

Analysez la complexite de recherche pour un probleme avec :
- 10 lieux
- 5 vehicules
- 3 colis a livrer

Comparez le nombre de noeuds a explorer en planification classique vs HTN.

In [10]:
# Exercice 2: Analyse de complexite

def analyze_complexity(num_locations, num_vehicles, num_packages):
    """
    Analyse la complexite de recherche pour les deux paradigmes.
    """
    print(f"Probleme: {num_locations} lieux, {num_vehicles} vehicules, {num_packages} colis")
    print("=" * 60)
    
    # Planification classique: espace d'etats
    # Estimation approximative du facteur de branchement
    # Actions possibles: load, unload, drive
    
    # Votre analyse ici...
    pass

analyze_complexity(10, 5, 3)

Probleme: 10 lieux, 5 vehicules, 3 colis


### Exercice 3 : Domaine de la cuisine

Creez un domaine HTN pour preparer un repas avec les taches :
- `prepare-meal` (tache abstraite)
- `cook-pasta`, `make-salad`, `serve` (taches abstraites)
- `boil-water`, `add-pasta`, `chop-vegetables`, `plate` (primitives)

In [11]:
# Exercice 3: Domaine cuisine HTN

# Votre code ici...

# Hint:
# - prepare-meal peut se decomposer en [cook-pasta, make-salad, serve]
# - cook-pasta peut se decomposer en [boil-water, add-pasta]
# - make-salad peut se decomposer en [chop-vegetables, plate]
# - Les actions primitives changent l'etat (water-boiled, pasta-cooked, etc.)

---

## 9. Resume et Points Cles

### 9.1 Concepts fondamentaux HTN

| Concept | Definition |
|---------|-------------|
| **Tache primitive** | Action directement executable |
| **Tache abstraite** | Tache a decomposer via methodes |
| **Methode** | Regle de decomposition avec preconditions |
| **Reseau de taches** | Ensemble ordonne de taches a accomplir |
| **SHOP2** | Algorithme de decomposition ordonnee |
| **HDDL** | Langage standard pour domains HTN |

### 9.2 Comparaison finale

```
+-------------------+----------------------+---------------------+
| Aspect            | Planification Class. | Planification HTN   |
+-------------------+----------------------+---------------------+
| Objectif          | Etat but             | Tache a accomplir   |
| Recherche         | Etats                | Decompositions      |
| Expertise         | Implicite            | Explicite (methodes)|
| Plans possibles   | Tous                 | Seulement prevus    |
| Optimalite        | Possible             | Non garantie        |
| Vitesse           | Variable             | Generalement rapide |
+-------------------+----------------------+---------------------+
```

### 9.3 Applications reelles

| Domaine | Usage HTN |
|---------|----------|
| **Robotique** | Plans de manipulation hierarchiques |
| **Logistique** | Chaines d'approvisionnement |
| **Jeux video** | IA de personnages (behaviour trees similaires) |
| **Operations militaires** | Plans d'operation hierarchiques |
| **Gestion de crise** | Procedures d'urgence |

---

## Ressources

### References academiques
- Erol, Hendler & Nau (1994) - "HTN Planning: Complexity and Expressivity"
- Nau et al. (2003) - "SHOP2: An HTN Planning System"
- Bercher et al. (2019) - "A Survey on Hierarchical Planning"

### Outils
- [SHOP2](https://github.com/shop-planner/shop2) - Planificateur HTN de reference
- [PANDA](https://www.uni-ulm.de/en/in/ap/institute-to/forschung/ai-planning/panda/) - Planificateur HTN allemand
- [unified-planning HTN](https://github.com/aiplan4eu/unified-planning) - Support HTN en Python

### Tutoriels
- [HTN Planning Tutorial](https://icaps20subpages.icaps-conference.org/tutorials/hierarchical-planning/)
- [HDDL Specification](https://github.com/panda-planner-dev/panda-framework)

---

**Notebook suivant** : [Planners-10-LLM-Planning](../04-NeuroSymbolic/Planners-10-LLM-Planning.ipynb)